# PV Module Lifetime and Waste-Flow Forecasting

**Project walkthrough:** Weibull lifetime fitting with right-censored observations, annual cohort retirements and waste mass estimation.

> The included data are synthetic. Outputs are methodological demonstrations, not Bangladesh national PV waste estimates.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pv_waste_forecasting.model import (
    fit_weibull_lifetime, survival_probability,
    forecast_cohort_waste
)

lifetime = pd.read_csv(ROOT / "data/demo/module_lifetime_observations_synthetic.csv")
cohorts = pd.read_csv(ROOT / "data/demo/pv_installation_cohorts_synthetic.csv")
settings = json.loads((ROOT / "config/base_case.json").read_text())

display(lifetime.head())
display(cohorts.head())

## 1. Fit a Weibull model with right-censored observations

In [ ]:
fit = fit_weibull_lifetime(
    lifetime["observed_lifetime_years"],
    lifetime["event_observed"]
)
fit

## 2. Survival curve and cohort retirement forecast

In [ ]:
ages = np.linspace(0, 45, 200)
survival = survival_probability(ages, fit["shape"], fit["scale_years"])

plt.figure(figsize=(8,4))
plt.plot(ages, survival)
plt.xlabel("Module age (years)")
plt.ylabel("Survival probability")
plt.title("Fitted Weibull survival curve (synthetic observations)")
plt.grid(alpha=.25)
plt.show()

years = np.arange(settings["forecast_start_year"], settings["forecast_end_year"]+1)
forecast = forecast_cohort_waste(
    cohorts, settings["base_year"], years, fit["shape"], fit["scale_years"]
)
display(forecast.head(10))

## 3. Interpret annual flows

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(forecast["year"], forecast["expected_module_waste_tonnes"])
ax.set(title="Expected annual module waste (synthetic demonstration)",
       xlabel="Year", ylabel="Tonnes")
plt.tight_layout()
plt.show()

forecast.to_csv(ROOT / "outputs/pv_waste_forecast_demo.csv", index=False)

## Research considerations

- Distinguish right-censoring from missing failure records.
- Separate technical retirement from collected and recycled waste.
- Add reuse/refurbishment and early-replacement pathways as explicit flow scenarios.
- Use documented Bangladesh installation cohorts and module mass data before empirical claims.
- Quantify parameter uncertainty and validate against independent observations.

The model output is conditional on the input cohort and lifetime assumptions.

## 4. Separate technical retirement from material pathways

The Weibull model estimates technical module retirement. The next step is to explore what may happen to that retired mass. Reuse, recycling and residual disposal are separate pathways and need separate evidence.

The shares in this example are illustrative. They do not represent measured Bangladesh collection or recycling rates.

In [ ]:
from pv_waste_forecasting.model import apply_material_pathways

pathways = apply_material_pathways(
    forecast,
    reuse_share=settings.get("reuse_share_demo", 0.0),
    recycling_share_of_nonreuse=settings.get("recycling_share_of_nonreuse_demo", 0.0),
    recycling_yield=settings.get("recycling_yield_demo", 0.0),
)
display(pathways.head(10).round(3))

assert np.allclose(pathways["mass_balance_residual_tonnes"], 0.0, atol=1e-9)
print("Mass balance closes arithmetically. This does not validate pathway feasibility.")

## 5. Lifetime sensitivity

Lifetime scale is an important assumption. A shorter scale implies earlier retirement; a longer scale implies later retirement. This simple sensitivity holds the fitted shape constant and changes the scale by ±10%. It is a scenario comparison, not a confidence interval.

In [ ]:
sensitivity_rows = []
for label, multiplier in [
    ("Shorter lifetime (-10%)", 0.90),
    ("Fitted base lifetime", 1.00),
    ("Longer lifetime (+10%)", 1.10),
]:
    case = forecast_cohort_waste(
        cohorts,
        settings["base_year"],
        years,
        fit["shape"],
        fit["scale_years"] * multiplier,
    )
    sensitivity_rows.append({
        "scenario": label,
        "total_expected_retirements_tonnes": case["expected_module_waste_tonnes"].sum(),
        "mean_annual_retirements_tonnes": case["expected_module_waste_tonnes"].mean(),
    })

lifetime_sensitivity = pd.DataFrame(sensitivity_rows)
display(lifetime_sensitivity.round(2))

ax = lifetime_sensitivity.plot(
    x="scenario",
    y="total_expected_retirements_tonnes",
    kind="bar",
    legend=False,
    rot=15,
    figsize=(8, 4),
    title="Lifetime-scale sensitivity (synthetic demonstration)",
)
ax.set_ylabel("Cumulative expected module waste (tonnes)")
plt.tight_layout()
plt.show()

## 6. Machine-learning degradation comparison

The new module compares Random Forest and Extra Trees using a holdout split by physical module ID. Grouping matters because multiple annual measurements from the same panel are correlated. A random row split could allow the same module to appear in training and test data, producing overly optimistic results.

The data below are synthetic and should not be used as a claim about Bangladesh degradation.

In [ ]:
from pv_waste_forecasting.degradation import compare_degradation_models

degradation_data = pd.read_csv(
    ROOT / "data/demo/module_degradation_panel_synthetic.csv"
)
degradation_result = compare_degradation_models(
    degradation_data,
    target_col="annual_degradation_rate",
)
display(degradation_result["comparison"].round(5))
display(degradation_result["feature_importance"].round(5))
print("Validation:", degradation_result["validation_method"])
print("Training modules:", len(degradation_result["train_module_ids"]))
print("Testing modules:", len(degradation_result["test_module_ids"]))
assert set(degradation_result["train_module_ids"]).isdisjoint(
    set(degradation_result["test_module_ids"])
)

## 7. How to move from this demo to research

Before replacing synthetic inputs:
1. Define a PV retirement event consistently.
2. Record module-level commissioning and last-observed dates for censoring.
3. Obtain enough observed retirements to estimate lifetime distributions.
4. Separate technologies or site types if they have distinct lifetime mechanisms and enough data.
5. Validate degradation models on unseen modules and, where possible, unseen sites or years.
6. Propagate uncertainty in Weibull parameters, installed capacity, module mass and material-pathway shares.

Do not report the demonstration MAE, RMSE or forecast tonnes as empirical Bangladesh findings.